# Rotten Fruit Detector — SageMaker Studio Training

Fine-tunes **SSD EfficientDet D5** on a fresh-vs-rotten fruit dataset using the code in `src/`.
Run this notebook from inside the cloned repo. Expected dataset layout:

```text
dataset/
  train/*.jpg
  train/annotations.csv
  valid/*.jpg
  valid/annotations.csv
  test/*.jpg
  test/annotations.csv
```


In [ ]:
# 1. Install dependencies
%pip install -q -r ../requirements.txt

In [ ]:
# 2. Download the dataset from S3 and unzip it
import boto3
from pathlib import Path

S3_BUCKET = "YOUR_BUCKET"
S3_KEY = "datasets/dataset.zip"
PROJECT_DIR = Path.cwd().parent

zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(S3_BUCKET, S3_KEY, str(zip_path))

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(PROJECT_DIR)

print(sorted(p.name for p in (PROJECT_DIR / "dataset").iterdir()))

In [ ]:
# 3. Train, evaluate, and export
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

from train import main

main(
    data_dir=str(PROJECT_DIR / "dataset"),
    num_steps=5000,
    batch_size=2,
)

In [ ]:
# 4. Upload the exported model back to S3
import sagemaker

session = sagemaker.Session()
uri = session.upload_data(
    path=str(PROJECT_DIR / "artifacts" / "ssd_efficientdet_d5_saved_model"),
    bucket=session.default_bucket(),
    key_prefix="rotten-fruit-detection/saved_model",
)
print("Uploaded to:", uri)